In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler
from sklearn import metrics

from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier


In [2]:
df = pd.read_csv("E1_datos.csv")
df

,Unnamed: 0,species,island,culmen_length_mm,culmen_depth_mm,flipper_length_mm,body_mass_g,sex
0,0,Adelie,Torgersen,39.1,18.7,181.0,3750.0,MALE
1,1,Adelie,Torgersen,39.5,17.4,186.0,3800.0,FEMALE
2,2,Adelie,Torgersen,40.3,18.0,195.0,3250.0,FEMALE
3,3,Adelie,Torgersen,NaN,NaN,NaN,NaN,NaN
4,4,Adelie,Torgersen,36.7,19.3,193.0,3450.0,FEMALE
...,...,...,...,...,...,...,...,...
339,339,Gentoo,Biscoe,NaN,NaN,NaN,NaN,NaN
340,340,Gentoo,Biscoe,46.8,14.3,215.0,4850.0,FEMALE
341,341,Gentoo,Biscoe,50.4,15.7,222.0,5750.0,MALE
342,342,Gentoo,Biscoe,45.2,14.8,212.0,5200.0,FEMALE


In [3]:
if "Unnamed: 0" in df.columns:
    df = df.drop(columns=["Unnamed: 0"])

### Mision 0: completando informacion
Abra el archivo y complete la informacion numerica faltante utilizando el valor promedio de cada columna.
A continuacion, descarte los registros para los cuales hay variables categoricas con valores faltantes.

In [4]:
columnas_numericas = df.select_dtypes(include=["int64", "float64"]).columns
for col in columnas_numericas:
    promedio = df[col].mean()
    df[col] = df[col].fillna(promedio)

# ahora eliminamos solo las filas que tengan valores faltantes en las columnas categoricas
columnas_categoricas = df.select_dtypes(include=["object"]).columns
columnas_categoricas


Index(['species', 'island', 'sex'], dtype='object')

In [5]:
df = df.dropna(subset=columnas_categoricas)

In [6]:
df[columnas_numericas].isna().sum()

culmen_length_mm     0
culmen_depth_mm      0
flipper_length_mm    0
body_mass_g          0
dtype: int64

In [7]:
df[columnas_categoricas].isna().sum()

species    0
island     0
sex        0
dtype: int64

Ahora buscamos separar las features del target para luego poder hacer el preprocesamiento de los datos.

In [8]:
target = "species"
X = df.drop(columns=[target])
y = df[target]

le = LabelEncoder()
y = le.fit_transform(y)

Viene entonces la parte del train y test.

In [9]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

`stratify` sirve para que la proporcion de clases en el train y test sea la misma que en el dataset original. 

Esto es especialmente importante cuando se tiene un dataset con clases desbalanceadas, ya que garantiza que ambas particiones tengan una representacion adecuada de cada clase.

Ahora podemos usar otro label encoder para codificar las features categoricas. 
En este caso, como tenemos mas de una feature categorica, es deseable usar `OneHotEncoder` para evitar que el modelo interprete un orden entre las categorias.

In [10]:
columnas_categoricas = ["island", "sex"]

ohe = OneHotEncoder(sparse_output=False, handle_unknown="ignore")

# hacemos el fit solo con el train para evitar data leakage, y luego transformamos ambos
X_train_cat = ohe.fit_transform(X_train[columnas_categoricas])
X_test_cat = ohe.transform(X_test[columnas_categoricas])

# nombres de columnas nuevas
nombres_ohe = ohe.get_feature_names_out(columnas_categoricas)
nombres_ohe

array(['island_Biscoe', 'island_Dream', 'island_Torgersen', 'sex_.',
       'sex_FEMALE', 'sex_MALE'], dtype=object)

In [11]:
# convertir a DataFrame
X_train_cat = pd.DataFrame(X_train_cat, columns=nombres_ohe, index=X_train.index)
X_test_cat = pd.DataFrame(X_test_cat, columns=nombres_ohe, index=X_test.index)

X_train_cat.head()

,island_Biscoe,island_Dream,island_Torgersen,sex_.,sex_FEMALE,sex_MALE
205,0.0,1.0,0.0,0.0,0.0,1.0
164,0.0,1.0,0.0,0.0,1.0,0.0
90,0.0,1.0,0.0,0.0,1.0,0.0
106,1.0,0.0,0.0,0.0,1.0,0.0
140,0.0,1.0,0.0,0.0,1.0,0.0


In [12]:
X_test_cat.head()

,island_Biscoe,island_Dream,island_Torgersen,sex_.,sex_FEMALE,sex_MALE
331,1.0,0.0,0.0,0.0,0.0,1.0
214,0.0,1.0,0.0,0.0,1.0,0.0
93,0.0,1.0,0.0,0.0,0.0,1.0
308,1.0,0.0,0.0,0.0,1.0,0.0
290,1.0,0.0,0.0,0.0,1.0,0.0


In [13]:
# eliminar originales y unir
X_train = X_train.drop(columns=columnas_categoricas).join(X_train_cat)
X_test = X_test.drop(columns=columnas_categoricas).join(X_test_cat)


In [14]:
X_train.head()

,culmen_length_mm,culmen_depth_mm,flipper_length_mm,body_mass_g,island_Biscoe,island_Dream,island_Torgersen,sex_.,sex_FEMALE,sex_MALE
205,50.7,19.7,203.0,4050.0,0.0,1.0,0.0,0.0,0.0,1.0
164,47.0,17.3,185.0,3700.0,0.0,1.0,0.0,0.0,1.0,0.0
90,35.7,18.0,202.0,3550.0,0.0,1.0,0.0,0.0,1.0,0.0
106,38.6,17.2,199.0,3750.0,1.0,0.0,0.0,0.0,1.0,0.0
140,40.2,17.1,193.0,3400.0,0.0,1.0,0.0,0.0,1.0,0.0


In [15]:
X_test.head()

,culmen_length_mm,culmen_depth_mm,flipper_length_mm,body_mass_g,island_Biscoe,island_Dream,island_Torgersen,sex_.,sex_FEMALE,sex_MALE
331,49.8,15.9,229.000000,5950.0,1.0,0.0,0.0,0.0,0.0,1.0
214,45.7,17.0,195.000000,3650.0,0.0,1.0,0.0,0.0,1.0,0.0
93,39.6,18.1,186.000000,4450.0,0.0,1.0,0.0,0.0,0.0,1.0
308,47.5,14.0,200.795455,4875.0,1.0,0.0,0.0,0.0,1.0,0.0
290,47.7,15.0,216.000000,4750.0,1.0,0.0,0.0,0.0,1.0,0.0


Ahora escalamos las numéricas.

In [16]:
columnas_numericas = ["culmen_length_mm", "culmen_depth_mm", "flipper_length_mm", "body_mass_g"]

scaler = StandardScaler()
X_train[columnas_numericas] = scaler.fit_transform(X_train[columnas_numericas])
X_test[columnas_numericas] = scaler.transform(X_test[columnas_numericas])

Hacemos la función de evaluación, y en este caso, como es un problema de clasificación binaria, usaremos `accuracy_score` para medir el desempeño del modelo.

In [17]:
def training_and_eval(model, X_train, X_test, y_train, y_test):
    model.fit(X_train, y_train)
    predictions = model.predict(X_test)
    accuracy = metrics.accuracy_score(y_test, predictions)
    return predictions, accuracy

### Mision 2: prediccion de la especie
Encuentre el mejor modelo para predecir la especie de un pinguino dadas sus caracterısticas. En particular,
debera evaluar dos posibles estrategias para construir modelos:

• Prediccion tradicional: entrenamiento de modelos para predecir directamente la raza de cada pinguino.

• Prediccion jerarquica: entrenamiento de dos modelos para predecir la raza del pinguino. El primero
debe discriminar entre 1 raza y las otras 2, mientras que el segundo debe discriminar entre las dos que
formaron el mismo grupo para el modelo anterior. Que raza usar para cada grupo y modelo es una
decision que debe tomar ud.


Partamos por la tradicional

In [18]:

model_tree = DecisionTreeClassifier(random_state=2115)
pred_tree, acc_tree = training_and_eval(model_tree, X_train, X_test, y_train, y_test)

model_svc = SVC()
pred_svc, acc_svc = training_and_eval(model_svc, X_train, X_test, y_train, y_test)

model_knn = KNeighborsClassifier()
pred_knn, acc_knn = training_and_eval(model_knn, X_train, X_test, y_train, y_test)

# buscar mejor modelo tradicional
resultados_tradicionales = {
    "Decision Tree": acc_tree,
    "SVC": acc_svc,
    "KNN": acc_knn
}

resultados_tradicionales


{'Decision Tree': 1.0, 'SVC': 0.9850746268656716, 'KNN': 0.9850746268656716}

In [19]:
mejor_modelo = max(resultados_tradicionales, key=resultados_tradicionales.get)
mejor_modelo

'Decision Tree'

In [20]:
mejor_acc = resultados_tradicionales[mejor_modelo]
mejor_acc

1.0

Vamos con la jerárquica. Primero, entonces, debemos decidir qué raza discriminar en el primer modelo.

In [21]:
print(f"Clases codificadas: {le.classes_}")

Clases codificadas: ['Adelie' 'Chinstrap' 'Gentoo']


In [22]:
df.groupby("species")[["culmen_length_mm", "culmen_depth_mm", "flipper_length_mm", "body_mass_g"]].mean()

,culmen_length_mm,culmen_depth_mm,flipper_length_mm,body_mass_g
species,,,,
Adelie,38.823973,18.347260,191.159869,3706.164384
Chinstrap,48.833824,18.420588,195.431818,3733.088235
Gentoo,47.542500,15.002500,215.654545,5090.625000


Vemos que en general, el más distinguible de las especies es Gentoo, así que vamos a discriminar entre Gentoo y el resto en el primer modelo.

Un diagrama para visualizar esto sería:

¿Es Gentoo?

│


├── Sí -> Gentoo

└── No -> ¿Adelie o Chinstrap?

De otro modo, tipo pseudocodigo:

```python
def predecir_pinguino(x):

    # Paso 1
    if modelo_1(x) == "Gentoo":
        return "Gentoo"

    # Paso 2
    else:
        return modelo_2(x)  # Adelie o Chinstrap
```

In [23]:
# capturar el índice de la clase Gentoo
idx_gentoo = np.where(le.classes_ == "Gentoo")[0][0]
idx_gentoo

2

Modelo 1: Gentoo vs no Gentoo

In [24]:
y_train_h1 = (y_train == idx_gentoo).astype(int)
y_test_h1 = (y_test == idx_gentoo).astype(int)

model_1 = DecisionTreeClassifier(random_state=42)
model_1.fit(X_train, y_train_h1)


,criterion,'gini'
,splitter,'best'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,None
,random_state,42
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,class_weight,None


Como es binario, si es 1 es Gentoo, y si es 0, es Adelie o Chinstrap.

Modelo 2: clasificar entre las otras dos especies
Primero hacemos un mask, que es un array q indica con True las filas que no son Gentoo, y con False las que sí lo son

In [25]:
mask_train_no_gentoo = y_train != idx_gentoo
print(mask_train_no_gentoo[:20])

[ True  True  True  True  True  True False  True  True False  True  True
 False  True  True  True False  True  True  True]


In [26]:
X_train_h2 = X_train[mask_train_no_gentoo]
y_train_h2 = y_train[mask_train_no_gentoo]

X_train_h2.head()

,culmen_length_mm,culmen_depth_mm,flipper_length_mm,body_mass_g,island_Biscoe,island_Dream,island_Torgersen,sex_.,sex_FEMALE,sex_MALE
205,1.231091,1.258406,0.155819,-0.217725,0.0,1.0,0.0,0.0,0.0,1.0
164,0.553205,0.046159,-1.213904,-0.652244,0.0,1.0,0.0,0.0,1.0,0.0
90,-1.517094,0.399731,0.079723,-0.838466,0.0,1.0,0.0,0.0,1.0,0.0
106,-0.985778,-0.004351,-0.148564,-0.590170,1.0,0.0,0.0,0.0,1.0,0.0
140,-0.692639,-0.054861,-0.605138,-1.024689,0.0,1.0,0.0,0.0,1.0,0.0


In [27]:
y_train_h2

array([1, 1, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0,
       1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1,
       0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 0, 1, 0, 0, 1, 1, 0, 0, 1,
       1, 1, 0, 0, 1, 1, 0, 0, 0, 1, 0, 1, 0, 0, 1, 1, 0, 0, 1, 1, 0, 0,
       0, 1, 1, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       1, 1, 0, 0, 0, 0, 0, 1, 0, 1, 1, 0, 0, 0, 0, 0, 1, 0, 0, 1, 1, 0,
       0, 1, 0, 0, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 1, 0,
       0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0])

In [28]:
model_2 = DecisionTreeClassifier(random_state=42)
model_2.fit(X_train_h2, y_train_h2)

,criterion,'gini'
,splitter,'best'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,None
,random_state,42
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,class_weight,None


Hacemos ahora la prediccion jerárquica final. 

In [29]:
pred_modelo_1 = model_1.predict(X_test)

predicciones_jerarquicas = []
for i in range(len(X_test)):
    if pred_modelo_1[i] == 1:
        predicciones_jerarquicas.append(idx_gentoo)
    else:
        fila = X_test.iloc[[i]]
        pred_final = model_2.predict(fila)[0]
        predicciones_jerarquicas.append(pred_final)

predicciones_jerarquicas = np.array(predicciones_jerarquicas)

acc_jerarquico = metrics.accuracy_score(y_test, predicciones_jerarquicas)
print(f"Accuracy del modelo jerárquico: {acc_jerarquico:.4f}")

Accuracy del modelo jerárquico: 0.9403


Por sanidad, se evalúan los modelos utilizando etiquetas aleatorias. Para ello, se permutan los valores de y_train, rompiendo la relación entre las features y el target. 

Si aun así el modelo presenta un accuracy alto, esto indica que no está aprendiendo la relación real entre las variables, lo que podría ser señal de data leakage o errores en el código. 

En cambio, si el accuracy cae a niveles cercanos al azar, es una buena señal de que el modelo original está capturando patrones relevantes en los datos.

In [30]:
y_train_random = np.random.permutation(y_train)

model_random = DecisionTreeClassifier(random_state=42)
model_random.fit(X_train, y_train_random)
pred_random = model_random.predict(X_test)
acc_random = metrics.accuracy_score(y_test, pred_random)

model_svc_random = SVC()
model_svc_random.fit(X_train, y_train_random)
pred_svc_random = model_svc_random.predict(X_test)
acc_svc_random = metrics.accuracy_score(y_test, pred_svc_random)

In [31]:
print(f"Accuracy con etiquetas aleatorias (Decision Tree): {acc_random:.4f}")
print(f"Accuracy con etiquetas aleatorias (SVC): {acc_svc_random:.4f}")

Accuracy con etiquetas aleatorias (Decision Tree): 0.3134
Accuracy con etiquetas aleatorias (SVC): 0.3881


#### Resultados consolidados

In [32]:
print("Accuracy Decision Tree:", acc_tree)
print("Accuracy SVC:", acc_svc)
print("Accuracy KNN:", acc_knn)
print("Mejor modelo tradicional:", mejor_modelo, "-", mejor_acc)
print("Accuracy jerárquico:", acc_jerarquico)

Accuracy Decision Tree: 1.0
Accuracy SVC: 0.9850746268656716
Accuracy KNN: 0.9850746268656716
Mejor modelo tradicional: Decision Tree - 1.0
Accuracy jerárquico: 0.9402985074626866


Mostramos algunas predicciones jerárquicas en texto

In [33]:
resultado = pd.DataFrame({
    "Real": le.inverse_transform(y_test),
    "Prediccion jerarquica": le.inverse_transform(predicciones_jerarquicas)
})

print("\nPrimeras predicciones del modelo jerárquico:")
print(resultado.head(10))


Primeras predicciones del modelo jerárquico:
        Real Prediccion jerarquica
0     Gentoo                Gentoo
1  Chinstrap             Chinstrap
2     Adelie                Adelie
3     Gentoo                Gentoo
4     Gentoo                Gentoo
5     Gentoo                Gentoo
6  Chinstrap             Chinstrap
7     Adelie                Adelie
8     Adelie                Adelie
9     Gentoo                Gentoo


In [34]:
resultado[resultado["Real"] != resultado["Prediccion jerarquica"]].head()

,Real,Prediccion jerarquica
32,Adelie,Chinstrap
39,Adelie,Chinstrap
47,Chinstrap,Gentoo
65,Adelie,Chinstrap
